# 02 | Huấn luyện mô hình V5: Random Forest 14 đặc trưng (thêm PPG_AC), đóng băng trên MIMIC

**Mục đích.** Tái tạo `models/mimic/healthsense_af_v5_rf_ac_frozen.pkl`, model đơn-domain được dùng làm mốc cho các kiểm định ngoài (DeepBeat, PulseWatch, Liu, PAC/PVC) ở `experiments/06_external_validation`. V5 là nơi đặc trưng hình thái sóng **PPG_AC** xuất hiện lần đầu, nâng số đặc trưng từ 13 lên 14.

Bản đọc được của `experiments/06_external_validation/train_frozen_model.py`.

| | |
|---|---|
| Đầu vào | `experiments/04_feature_model/outputs/af_features_cosen_ac.csv`: 14 đặc trưng của MIMIC PERform AF, có cột `record_id`, `status` |
| Đầu ra | `models/mimic/healthsense_af_v5_rf_ac_frozen.pkl` và `.json` |
| Cách chọn tham số | GridSearchCV trên lưới Random Forest trong `healthsense_ml.training.build_model_registry`, cross-validation GroupKFold 3 fold **theo bệnh nhân**, điểm ROC-AUC |

In [1]:
# Tìm gốc repo (chạy được từ notebooks/training/ hoặc từ gốc) và nạp thư viện src/healthsense_ml
import os, sys, json, time
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "src" / "healthsense_ml").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("Không tìm thấy gốc repo (thư mục chứa src/healthsense_ml)")
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
print("Repo root:", ROOT)

import numpy as np
import pandas as pd
import joblib
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

Repo root: D:\Projects\HealthSense\HealthSense-MachineLearning-Lab


In [2]:
from sklearn.base import clone
from sklearn.model_selection import GroupKFold, GridSearchCV
from healthsense_ml.training import build_model_registry

DATA = ROOT / "experiments" / "04_feature_model" / "outputs" / "af_features_cosen_ac.csv"
OUT_DIR = ROOT / "models" / "mimic"
FEATURES = ["HR_mean", "Mean_NN", "SDNN", "RMSSD", "NN50", "pNN50", "CV",
            "HF", "Total_Power", "HF_norm", "SD1", "SD2", "SampEn", "PPG_AC"]

df = pd.read_csv(DATA)
X, y, groups = df[FEATURES], df["status"].to_numpy(), df["record_id"].to_numpy()
print(f"{len(df)} cửa sổ | {df.record_id.nunique()} bệnh nhân | tỉ lệ AF {y.mean():.2%}")
df[FEATURES].describe().T[["mean", "std", "min", "max"]].round(3)

4130 cửa sổ | 35 bệnh nhân | tỉ lệ AF 54.29%


,mean,std,min,max
HR_mean,87.706,18.692,43.940,132.035
Mean_NN,718.802,168.613,454.424,1365.500
SDNN,127.043,96.026,4.316,564.133
RMSSD,168.541,126.942,5.950,785.779
NN50,22.011,14.873,0.000,52.000
pNN50,52.014,32.661,0.000,100.000
CV,0.182,0.128,0.008,0.617
HF,11118.117,27046.153,0.721,528586.874
Total_Power,28471.406,217192.068,2.289,9737581.200
HF_norm,73.216,18.364,0.571,100.000


## 1. Dò siêu tham số theo bệnh nhân

GroupKFold với nhóm là `record_id` bảo đảm các cửa sổ của cùng một bệnh nhân không nằm ở cả hai phía train và validation trong lúc dò tham số. Lưới tham số lấy từ registry chung của thư viện để mọi thí nghiệm dùng cùng một lưới.

In [3]:
rf_spec = build_model_registry()["Random Forest"]
print("Lưới tham số:", rf_spec["param_grid"])

search = GridSearchCV(clone(rf_spec["pipeline"]), rf_spec["param_grid"],
                      cv=GroupKFold(n_splits=3), scoring="roc_auc", n_jobs=-1, refit=True)
t0 = time.time()
search.fit(X, y, groups=groups)
print(f"Xong trong {time.time()-t0:.0f}s")
print("Tham số tốt nhất:", search.best_params_)
print("ROC-AUC grouped-CV:", round(search.best_score_, 4))

Lưới tham số: {'clf__n_estimators': [100, 150], 'clf__max_depth': [5, 7, 10]}


Xong trong 10s
Tham số tốt nhất: {'clf__max_depth': 10, 'clf__n_estimators': 100}
ROC-AUC grouped-CV: 0.9903


## 2. Đóng băng model và ghi metadata

Model được `refit` trên toàn bộ MIMIC với tham số tốt nhất. Metadata ghi rõ đây là ước lượng nội bộ, phải kiểm định ngoài trước khi triển khai; các chỉ số nhạy, đặc hiệu mức bệnh nhân lấy từ vòng nested đánh giá trước đó của lab.

In [4]:
OVERWRITE_FROZEN = False   # True: ghi đè đúng tên file đã đóng băng (chỉ khi cố ý tạo lại bản chuẩn)
OUT_DIR.mkdir(parents=True, exist_ok=True)
stem = "healthsense_af_v5_rf_ac_frozen" if OVERWRITE_FROZEN else "healthsense_af_v5_rf_ac_retrained"
model_path = OUT_DIR / f"{stem}.pkl"
joblib.dump(search.best_estimator_, model_path)
meta = {
    "version": "healthsense-af-v5", "development_dataset": "MIMIC PERform AF",
    "subjects": int(df.record_id.nunique()), "features": FEATURES, "n_features": len(FEATURES),
    "window_s": 30, "step_s": 10, "iqr_filter": False, "calibration": None,
    "alert_threshold": 0.90, "alert_policy": "3-of-3",
    "best_params": search.best_params_, "grouped_cv_auc": round(float(search.best_score_), 4),
    "internal_nested_subject_sensitivity": 1.0, "internal_nested_subject_specificity": 0.75,
    "note": "Internal metrics are development estimates. External validation required before deployment.",
}
json.dump(meta, open(OUT_DIR / f"{stem}.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print("Đã lưu:", model_path, f"({model_path.stat().st_size/1024:.0f} KB)")

Đã lưu: D:\Projects\HealthSense\HealthSense-MachineLearning-Lab\models\mimic\healthsense_af_v5_rf_ac_retrained.pkl (1065 KB)
